[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1m0Oi8UKgX2HyWkz2w4qfXnI5y6iWD3Ni/view?usp=drive_link)

# Agent Evaluation – Local Agent

This notebook demonstrates two ways to evaluate a local Python agent:

1. **LangChain agent** — Build an agent with LangChain tools and wrap it with `wrap_langchain_agent` for automatic trace capture.
2. **Custom agent with `@capture_trace`** — Use `log_turn` and `log_tool_result` to manually record what happens during execution.

Floeval runs your agent for each test case, captures the trace, and scores it.

**Objectives**
- Install Floeval and configure credentials
- Define a LangChain agent with tools and wrap it with `wrap_langchain_agent`
- Define a custom agent with `@capture_trace` and manual trace logging
- Load partial agent datasets from JSON files you provide and run evaluations for both

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install floeval>=0.2.0b1

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass
# LLM and API configuration (OpenAI)

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

Import the agent evaluation components, dataset schemas, and trace helpers: `capture_trace`, `log_turn`, `log_tool_result`, and `wrap_langchain_agent`.

In [ ]:
from pathlib import Path

from floeval.api.agent_evaluation import AgentEvaluation
from floeval.api.dataset_loaders.agent_file_loader import AgentDatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig
from floeval.utils.agent_trace import capture_trace, log_tool_result, log_turn, wrap_langchain_agent

## 4. Agent A: LangChain Agent (auto trace capture)

Build an agent with LangChain tools. `wrap_langchain_agent` captures traces automatically via LangChain callbacks — no manual logging needed.

**Step 1:** Define tools with the `@tool` decorator. Each tool has a docstring that the LLM uses to decide when to call it.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


@tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    db = {"paris": "18°C", "tokyo": "22°C", "london": "14°C"}
    return db.get(city.lower(), f"{city}: No data")

**Step 2:** Create the LLM and agent. `create_agent` builds a graph-based agent that calls the model and tools in a loop until it produces a final answer.

In [ ]:
llm_langchain = ChatOpenAI(
    model=OPENAI_CHAT_MODEL,
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)
langchain_agent = create_agent(
    model=llm_langchain,
    tools=[calculator, get_weather],
    system_prompt="Use tools when needed. Answer accurately.",
)

**Step 3:** Wrap the agent with `wrap_langchain_agent`. This adapts the LangChain agent to Floeval's interface (string in, string out) and enables automatic trace capture via LangChain callbacks.

In [ ]:
wrapped_langchain_agent = wrap_langchain_agent(langchain_agent)

## 5. Agent B: Custom Agent with `@capture_trace` (manual trace logging)

For custom Python agents (not LangChain), use `@capture_trace` and manually log what happens. Call `log_tool_result` after each tool execution and `log_turn` for each AI response. The agent must accept `user_input` (str) and return a string.

In [ ]:
@capture_trace
def support_agent(user_input: str) -> str:
    """Simple support agent that simulates a ticket lookup."""
    tool_output = f"Ticket #123 created for {user_input}. We will follow up within 24 hours."
    log_tool_result("create_ticket", tool_output)
    final_text = f"Request received. {tool_output}"
    log_turn(final_text)
    return final_text

## 6. Load Partial Datasets (JSON)

Minimal JSON shape (**partial** - no `trace`; Floeval runs your agent):

```json
{
  "samples": [
    {
      "user_input": "...",
      "reference_outcome": "...",
      "reference_tool_calls": [{ "name": "...", "args": {} }]
    }
  ]
}
```

**Example files**  
<a href="../datasets/agent_evaluation/sample_agent_partial_langchain.json" download="sample_agent_partial_langchain.json">sample_agent_partial_langchain.json</a><br>
<a href="../datasets/agent_evaluation/sample_agent_partial_support.json" download="sample_agent_partial_support.json">sample_agent_partial_support.json</a>

Provide paths for both dataset files and load them with `AgentDatasetLoader.from_file(...)`.


### Resolve path (LangChain dataset)

Upload or paste the path to the **LangChain** partial agent JSON (`path_langchain`).


In [ ]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload LangChain partial agent JSON file:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    path_langchain = Path(next(iter(uploaded.keys())))
else:
    path_langchain = Path(input("Enter path to LangChain partial agent JSON file: ").strip().strip('"')).expanduser()


### Load LangChain dataset

`AgentDatasetLoader.from_file(path_langchain)` — tool names should match the LangChain tools above.


In [ ]:
dataset_langchain = AgentDatasetLoader.from_file(path_langchain)
print(f"LangChain dataset from {path_langchain}: {len(dataset_langchain.samples)} sample(s)")


**Dataset for support agent:** Support ticket queries. No `reference_tool_calls` needed for basic metrics.

### Resolve path (support dataset)

Second upload/prompt: path to the **support** partial agent JSON (`path_support`).


In [ ]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload support partial agent JSON file:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    path_support = Path(next(iter(uploaded.keys())))
else:
    path_support = Path(input("Enter path to support partial agent JSON file: ").strip().strip('"')).expanduser()


### Load support dataset

`AgentDatasetLoader.from_file(path_support)` for the custom `@capture_trace` scenarios.


In [ ]:
dataset_support = AgentDatasetLoader.from_file(path_support)
print(f"Support agent dataset from {path_support}: {len(dataset_support.samples)} sample(s)")


## 7. Configure the LLM

Create the LLM configuration used by Floeval for scoring metrics (goal_achievement, response_coherence, etc.). The agent uses its own LLM for generation.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 8. Run Evaluation: LangChain Agent

Pass `wrapped_langchain_agent` to `AgentEvaluation`. Floeval runs the agent for each sample, captures the trace automatically, and scores it. We include `ragas:tool_call_accuracy` since we have `reference_tool_calls`.

### Build LangChain `AgentEvaluation`

Wires `wrapped_langchain_agent`, optional `ragas:tool_call_accuracy` when references exist.


In [ ]:
eval_langchain = AgentEvaluation(
    dataset=dataset_langchain,
    agent=wrapped_langchain_agent,
    llm_config=llm_config,
    metrics=["goal_achievement", "response_coherence", "ragas:tool_call_accuracy"],
    default_provider="builtin",
)


### Run LangChain evaluation

`eval_langchain.run()` produces `results_langchain` (summary + sample_results).


In [ ]:
results_langchain = eval_langchain.run()


## 9. Run Evaluation: Support Agent

Pass `support_agent` to `AgentEvaluation`. Floeval runs the agent, captures the trace via `@capture_trace`, and scores it.

### Build support `AgentEvaluation`

Uses `support_agent` with goal/coherence metrics (no tool-accuracy requirement for this demo).


In [ ]:
eval_support = AgentEvaluation(
    dataset=dataset_support,
    agent=support_agent,
    llm_config=llm_config,
    metrics=["goal_achievement", "response_coherence"],
    default_provider="builtin",
)


### Run support evaluation

`eval_support.run()` produces `results_support`.


In [ ]:
results_support = eval_support.run()


## 10. Display Results

**Aggregate scores:** Mean score per metric across all samples. **Per-sample results:** Each row shows the captured `final_response` and individual metric scores.

### Display aggregate scores

Defines `_display_summary` and prints mean metric scores for both agents.


In [ ]:
def _display_summary(name: str, summary: dict) -> None:
    """Display aggregate scores in a readable format."""
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    for metric, score in summary.items():
        if isinstance(score, float):
            print(f"  {metric}: {score:.3f}")
        else:
            print(f"  {metric}: {score}")
    print()

_display_summary("LangChain Agent — Aggregate Scores", results_langchain.summary)
_display_summary("Support Agent — Aggregate Scores", results_support.summary)

**Per-sample results:** Inspect each sample's final response and metric scores.

### Display per-sample results

Defines `_display_sample_results` for side-by-side inspection of inputs, responses, and metric scores.


In [ ]:
def _display_sample_results(name: str, sample_results: list) -> None:
    """Display per-sample results in a readable format."""
    print(f"\n--- {name} ---")
    for i, sr in enumerate(sample_results, start=1):
        user_in = sr.get("user_input", "")
        final = sr.get("final_response") or ""
        user_disp = user_in[:60] + ("..." if len(user_in) > 60 else "")
        final_disp = final[:80] + ("..." if len(final) > 80 else "")
        print(f"\n  Sample {i}")
        print(f"    Input:    {user_disp}")
        print(f"    Response: {final_disp}")
        for k, v in sr.get("metrics", {}).items():
            score = v.get("score")
            disp = f"{score:.3f}" if isinstance(score, float) else score
            print(f"    {k}: {disp}")

_display_sample_results("LangChain Agent", results_langchain.sample_results)
_display_sample_results("Support Agent", results_support.sample_results)

## Summary

This notebook demonstrated how to evaluate two local Python agents: a LangChain agent with automatic trace capture and a custom agent with manual trace logging.

The key components included:

1. **LangChain Agent**: An agent was built with `create_agent`, calculator and weather tools, and wrapped with `wrap_langchain_agent` so traces are captured via LangChain callbacks.
2. **Custom Agent**: `support_agent` used `@capture_trace` with `log_tool_result` and `log_turn` to record the trace manually.
3. **Datasets**: Partial agent datasets were loaded from JSON for each agent and matched their tools.
4. **Evaluation**: `AgentEvaluation` was run for both agents, including `ragas:tool_call_accuracy` where reference tool calls were provided.
5. **Contract**: Both agents accept `user_input` (str) and return a string (or `AgentTrace`).

This example showcases local agent evaluation with Floeval trace capture patterns.